In [1]:
import sys
sys.path.append("../src")
from data_prep import process_subject_session

windows, labels = process_subject_session(
    "../data/ds005931/sub-01/ses-01/ieeg/sub-01_ses-01_task-game_run-01_ieeg.edf",
    "../data/ds005931/sub-01/ses-01/ieeg/sub-01_ses-01_task-game_run-01_channels.tsv"
)

print("windows shape:", windows.shape)
print("labels shape:", labels.shape)
print("movement fraction:", labels.mean())
print("channels kept:", windows.shape[1])

windows shape: (2185, 19, 50)
labels shape: (2185,)
movement fraction: 0.16384439359267736
channels kept: 19


In [2]:
import glob

edf_files = sorted(glob.glob("../data/ds005931/sub-01/ses-*/ieeg/*_ieeg.edf"))
print(edf_files)

['../data/ds005931/sub-01/ses-01/ieeg/sub-01_ses-01_task-game_run-01_ieeg.edf', '../data/ds005931/sub-01/ses-02/ieeg/sub-01_ses-02_task-game_run-01_ieeg.edf']


In [3]:
from data_prep import process_subject_session

all_windows = []
all_labels = []

for edf_path in edf_files:
    channels_path = edf_path.replace("_ieeg.edf", "_channels.tsv")
    print(f"Processing {edf_path}...")
    w, l = process_subject_session(edf_path, channels_path)
    print(f"  -> {w.shape[0]} windows, {w.shape[1]} channels, movement frac {l.mean():.3f}")
    all_windows.append(w)
    all_labels.append(l)

print("Channel counts per session:", [w.shape[1] for w in all_windows])

Processing ../data/ds005931/sub-01/ses-01/ieeg/sub-01_ses-01_task-game_run-01_ieeg.edf...
  -> 2185 windows, 19 channels, movement frac 0.164
Processing ../data/ds005931/sub-01/ses-02/ieeg/sub-01_ses-02_task-game_run-01_ieeg.edf...
  -> 2185 windows, 64 channels, movement frac 0.164
Channel counts per session: [19, 64]


In [4]:
import importlib
import data_prep
importlib.reload(data_prep)   # picks up the new functions without restarting the kernel
from data_prep import get_common_good_channels, process_subject_session_fixed

session_paths = [(p, p.replace("_ieeg.edf", "_channels.tsv")) for p in edf_files]
common_channels = get_common_good_channels(session_paths)
print(f"Common good channels across sessions: {len(common_channels)}")
print(sorted(common_channels))

Common good channels across sessions: 0
[]


In [7]:
for edf_path, channels_path in session_paths:
    good = load_good_channels(channels_path)
    raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
    name_map = {clean_channel_name(ch): ch for ch in raw.ch_names}
    good_in_edf = {n for n in good if n in name_map}
    print(edf_path)
    print(f"  good (from tsv): {len(good)}  -> {sorted(good)[:10]}...")
    print(f"  good AND in EDF: {len(good_in_edf)} -> {sorted(good_in_edf)[:10]}...")
    print(f"  all EDF channel names (cleaned): {sorted(name_map.keys())[:10]}...")

../data/ds005931/sub-01/ses-01/ieeg/sub-01_ses-01_task-game_run-01_ieeg.edf
  good (from tsv): 83  -> ['A2', 'A27', 'A28', 'A29', 'A3', 'A37', 'A38', 'A39', 'A4', 'A40']...
  good AND in EDF: 19 -> ['A2', 'A27', 'A28', 'A29', 'A3', 'A37', 'A38', 'A39', 'A4', 'A40']...
  all EDF channel names (cleaned): ['A1', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18']...
../data/ds005931/sub-01/ses-02/ieeg/sub-01_ses-02_task-game_run-01_ieeg.edf
  good (from tsv): 83  -> ['A2', 'A27', 'A28', 'A29', 'A3', 'A37', 'A38', 'A39', 'A4', 'A40']...
  good AND in EDF: 64 -> ['B1', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18']...
  all EDF channel names (cleaned): ['B1', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18']...


In [6]:
from data_prep import (
    process_subject_session,
    get_common_good_channels,
    process_subject_session_fixed,
    load_good_channels,
    clean_channel_name,
)
import mne

In [8]:
from data_prep import process_subject_session

edf_path = "../data/ds005931/sub-01/ses-02/ieeg/sub-01_ses-02_task-game_run-01_ieeg.edf"
channels_path = edf_path.replace("_ieeg.edf", "_channels.tsv")

windows, labels = process_subject_session(edf_path, channels_path)
print("windows shape:", windows.shape)
print("labels shape:", labels.shape)
print("movement fraction:", labels.mean())

windows shape: (2185, 64, 50)
labels shape: (2185,)
movement fraction: 0.16384439359267736


In [9]:
import numpy as np

np.savez("../data/processed_sub01_ses02.npz", windows=windows, labels=labels)
print("Saved.")

Saved.


In [10]:
import importlib
import data_prep
importlib.reload(data_prep)
from data_prep import load_subject_session, get_movement_mask_v2, windowize_with_features

edf_path = "../data/ds005931/sub-01/ses-02/ieeg/sub-01_ses-02_task-game_run-01_ieeg.edf"
channels_path = edf_path.replace("_ieeg.edf", "_channels.tsv")

raw = load_subject_session(edf_path, channels_path)
mask = get_movement_mask_v2(raw)
windows, band_features, labels = windowize_with_features(raw, mask)

print(windows.shape, band_features.shape, labels.shape, labels.mean())

np.savez("../data/processed_sub01_ses02_v2.npz", windows=windows, band_features=band_features, labels=labels)

(2185, 64, 50) (2185, 64, 4) (2185,) 0.18352402745995422


In [11]:
move_gamma = band_features[labels == 1, :, 3].mean()
rest_gamma = band_features[labels == 0, :, 3].mean()

from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(band_features[labels == 1, :, 3].flatten(), band_features[labels == 0, :, 3].flatten())

print("movement gamma:", move_gamma)
print("rest gamma:", rest_gamma)
print("percent difference:", (move_gamma - rest_gamma) / rest_gamma * 100)
print("p-value:", p)

movement gamma: 5.6355253e-05
rest gamma: 6.131118e-05
percent difference: -8.083231
p-value: 7.1368193e-12


In [12]:
import numpy as np

per_channel_diff = []
for ch in range(64):
    move_ch = band_features[labels == 1, ch, 3]
    rest_ch = band_features[labels == 0, ch, 3]
    pct = (move_ch.mean() - rest_ch.mean()) / rest_ch.mean() * 100
    per_channel_diff.append(pct)

per_channel_diff = np.array(per_channel_diff)
top_idx = np.argsort(np.abs(per_channel_diff))[::-1][:10]

for i in top_idx:
    print(f"channel {i}: {per_channel_diff[i]:.2f}%")

channel 43: 184.23%
channel 0: -98.96%
channel 23: 45.45%
channel 1: 39.41%
channel 50: 38.61%
channel 5: -36.29%
channel 42: 30.31%
channel 40: 24.15%
channel 9: 22.98%
channel 63: 22.29%


In [13]:
for i in [43, 0, 23, 1, 50]:
    move_ch = band_features[labels == 1, i, 3]
    rest_ch = band_features[labels == 0, i, 3]
    print(f"channel {i}: move_mean={move_ch.mean():.6e}, rest_mean={rest_ch.mean():.6e}, "
          f"move_std={move_ch.std():.6e}, rest_std={rest_ch.std():.6e}")

channel 43: move_mean=2.180574e-04, rest_mean=7.671769e-05, move_std=2.965255e-04, rest_std=7.656426e-05
channel 0: move_mean=6.423359e-06, rest_mean=6.157466e-04, move_std=1.057884e-05, rest_std=9.173152e-03
channel 23: move_mean=3.148643e-05, rest_mean=2.164742e-05, move_std=7.919235e-05, rest_std=2.059136e-05
channel 1: move_mean=1.672911e-04, rest_mean=1.199973e-04, move_std=1.521734e-04, rest_std=1.063277e-04
channel 50: move_mean=1.820235e-04, rest_mean=1.313199e-04, move_std=1.809741e-04, rest_std=1.728406e-04


In [14]:
import csv
with open(channels_path) as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        if row["name"] in ["A44", "B44", "A1", "B1"]:
            print(row)

{'name': 'A1', 'type': 'ECOG', 'units': 'µV', 'low_cutoff': '0.016', 'high_cutoff': '300', 'reference': 'n/a', 'group': 'n/a', 'sampling_frequency': '1000', 'description': 'n/a', 'notch': 'n/a', 'status': 'bad', 'status_description': 'artifact'}
{'name': 'A44', 'type': 'ECOG', 'units': 'µV', 'low_cutoff': '0.016', 'high_cutoff': '300', 'reference': 'n/a', 'group': 'n/a', 'sampling_frequency': '1000', 'description': 'n/a', 'notch': 'n/a', 'status': 'good', 'status_description': 'n/a'}
{'name': 'B1', 'type': 'ECOG', 'units': 'µV', 'low_cutoff': '0.016', 'high_cutoff': '300', 'reference': 'n/a', 'group': 'n/a', 'sampling_frequency': '1000', 'description': 'n/a', 'notch': 'n/a', 'status': 'good', 'status_description': 'n/a'}
{'name': 'B44', 'type': 'ECOG', 'units': 'µV', 'low_cutoff': '0.016', 'high_cutoff': '300', 'reference': 'n/a', 'group': 'n/a', 'sampling_frequency': '1000', 'description': 'n/a', 'notch': 'n/a', 'status': 'good', 'status_description': 'n/a'}


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

X = band_features.reshape(band_features.shape[0], -1)
y = labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_train, y_train)

acc = accuracy_score(y_test, clf.predict(X_test))
auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

print("accuracy:", acc)
print("AUC:", auc)
print("baseline accuracy (majority class):", 1 - y.mean())

accuracy: 0.6051829268292683
AUC: 0.740625
baseline accuracy (majority class): 0.8164759725400458
